# Principio Abierto/Cerrado (OCP) — Cafetería

**Dominio propio:** cálculo de descuentos/promociones en los pedidos de la cafetería.

El OCP dice que una entidad debe estar **abierta a extensión pero cerrada a modificación**: agregar una promoción nueva no debería obligarme a editar la clase que ya funciona.

Muestro primero la versión con `if/elif` que crece (viola) y luego la versión con abstracción polimórfica (cumple).

## Versión que VIOLA el OCP

El cálculo del descuento vive dentro de un `if/elif` en `CalculadoraDescuento`. Cada promoción nueva (fiesta, empleado, etc.) obliga a **modificar** el método `aplicar`.

In [1]:
class CalculadoraDescuento:
    def __init__(self, tipo: str) -> None:
        self.tipo: str = tipo
        self.historial: list[float] = []

    def aplicar(self, total: float) -> float:
        # Cada promocion nueva obliga a tocar este metodo (viola OCP)
        if self.tipo == "ninguno":
            descuento = 0.0
        elif self.tipo == "estudiante":
            descuento = total * 0.10
        elif self.tipo == "hora_feliz":
            descuento = total * 0.20
        else:
            descuento = 0.0
        final = total - descuento
        self.historial.append(final)
        return final

    def total_acumulado(self) -> float:
        return sum(self.historial)

In [2]:
print(CalculadoraDescuento("estudiante").aplicar(10000))  # 9000.0
print(CalculadoraDescuento("hora_feliz").aplicar(10000))  # 8000.0

9000.0
8000.0


### Problema

Para agregar una promoción "empleado 30%" tengo que **editar** el `if/elif` de `aplicar`. Cada cambio arriesga romper las promociones ya probadas: la clase no está cerrada a modificación.

## Versión que CUMPLE el OCP

Defino una abstracción `Promocion` y cada promoción es una subclase. La `CalculadoraDescuento` trabaja contra la abstracción, así agrego promociones **sin tocar** su código.

In [1]:
from abc import ABC, abstractmethod

class Promocion(ABC):
    def __init__(self, nombre: str) -> None:
        self.nombre: str = nombre
        self.veces_aplicada: int = 0

    @abstractmethod
    def porcentaje(self) -> float:
        ...

    def aplicar(self, total: float) -> float:
        self.veces_aplicada += 1
        return total - total * self.porcentaje()


class SinPromocion(Promocion):
    def __init__(self) -> None:
        super().__init__("Sin promocion")

    def porcentaje(self) -> float:
        return 0.0


class PromoEstudiante(Promocion):
    def __init__(self) -> None:
        super().__init__("Estudiante")

    def porcentaje(self) -> float:
        return 0.10


class PromoHoraFeliz(Promocion):
    def __init__(self) -> None:
        super().__init__("Hora feliz")

    def porcentaje(self) -> float:
        return 0.20


class CalculadoraDescuento:
    """Cerrada a modificacion: no cambia aunque se agreguen promociones."""
    def __init__(self) -> None:
        self.historial: list[float] = []

    def aplicar(self, promocion: Promocion, total: float) -> float:
        final = promocion.aplicar(total)
        self.historial.append(final)
        return final

    def total_acumulado(self) -> float:
        return sum(self.historial)

In [4]:
calc = CalculadoraDescuento()
print(calc.aplicar(PromoEstudiante(), 10000))  # 9000.0
print(calc.aplicar(PromoHoraFeliz(), 10000))   # 8000.0

9000.0
8000.0


### Extensión sin modificar (la prueba del OCP)

Agrego una promoción nueva `PromoEmpleado` creando **solo una clase nueva**. No toco `Promocion` ni `CalculadoraDescuento`.

In [5]:
class PromoEmpleado(Promocion):
    def __init__(self) -> None:
        super().__init__("Empleado")

    def porcentaje(self) -> float:
        return 0.30

# Funciona con la misma calculadora, sin modificarla
print(calc.aplicar(PromoEmpleado(), 10000))  # 7000.0
print("Total acumulado:", calc.total_acumulado())

7000.0
Total acumulado: 24000.0


### Análisis

El comportamiento se extiende agregando subclases de `Promocion`. La `CalculadoraDescuento` quedó **cerrada a modificación** (no cambia) y el sistema **abierto a extensión** (nuevas promociones entran como clases nuevas).